# TRIBE V2 — Simple pre-render

**Before running:** Runtime → Change runtime type → T4 GPU

Run cells top to bottom. Each video takes ~5–10 min on T4.

In [ ]:
# 1. Confirm GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies (~3 min) — kernel will auto-restart after this cell
# numpy must be pinned to <2.0 — tribev2 uses internal APIs removed in numpy 2.x
!pip install -q "numpy<2.0"
!pip install -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"
!pip install -q nilearn nibabel
import os
os.kill(os.getpid(), 9)  # auto-restart so numpy downgrade takes effect

In [ ]:
# 3. HuggingFace login
# Needs read access to facebook/tribev2 and meta-llama/Llama-3.2-3B (gated)
# Get token at: https://huggingface.co/settings/tokens
from huggingface_hub import login
login()

In [ ]:
# 4. Upload aggregate.py and atlas.py from your junsoo folder
from google.colab import files
print('Upload aggregate.py and atlas.py from junsoo/')
files.upload()

In [ ]:
# 5. Upload all your video files at once
from google.colab import files
print('Upload your video files (.mp4 etc) — you can select multiple')
uploaded = files.upload()
video_paths = list(uploaded.keys())
print(f'Uploaded {len(video_paths)} video(s):')
for v in video_paths:
    print(f'  {v}')

In [ ]:
# 6. Run TRIBE V2 inference on all uploaded videos
from pathlib import Path
from tribev2.demo_utils import TribeModel
import numpy as np
import pickle
import json

cache = Path('./cache')
cache.mkdir(exist_ok=True)

print('Loading model...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder=cache)

for video_path in video_paths:
    stem = Path(video_path).stem
    print(f'\n==> [{stem}] Running inference...')

    df = model.get_events_dataframe(video_path=video_path)
    preds, segments = model.predict(events=df)
    print(f'    preds shape: {preds.shape}')

    np.savez(f'{stem}_preds.npz', preds=preds.astype('float32'))
    with open(f'{stem}_preds.segments.pkl', 'wb') as f:
        pickle.dump(segments, f)

    print(f'    Aggregating to {stem}_activity.json...')
    import subprocess
    result = subprocess.run(
        ['python', 'aggregate.py',
         '--preds', f'{stem}_preds.npz',
         '--segments', f'{stem}_preds.segments.pkl',
         '--out', f'{stem}_activity.json',
         '--atlas', 'yeo7',
         '--cache', './cache'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'    ERROR: {result.stderr}')
    else:
        with open(f'{stem}_activity.json') as f:
            d = json.load(f)
        print(f'    Done: {d["n_timesteps"]} frames')

print('\nAll videos processed!')

In [ ]:
# 7. Download all activity.json files
from google.colab import files
import glob

json_files = glob.glob('*_activity.json')
print(f'Downloading {len(json_files)} file(s)...')
for f in json_files:
    print(f'  {f}')
    files.download(f)
print('Done! Place each file in junsoo/prerendered/<clip_name>/activity.json')